In [ ]:
# === Setup (Part 2) — keep byte-identical across RQ notebooks ===
import json
import os
import pickle
from pathlib import Path

import torch
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer

PROJECT_ROOT = Path.home() / "projects" / "Measuring-Semantic-Stability-in-Clinical-LLMs"
CONFIG_PATH = PROJECT_ROOT / "config.json"
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"CONFIG_PATH:  {CONFIG_PATH}")
assert CONFIG_PATH.is_file(), f"Missing config.json: {CONFIG_PATH}"

with open(CONFIG_PATH, "r", encoding="utf-8") as _f:
    CFG = json.load(_f)


def _expand_tree(obj):
    """Expand ~ in all string paths; leave non-strings / null unchanged."""
    if isinstance(obj, dict):
        return {k: _expand_tree(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_expand_tree(v) for v in obj]
    if isinstance(obj, str):
        return os.path.expanduser(obj)
    return obj


CFG = _expand_tree(CFG)


def _resolve_cfg_path(p):
    if p is None:
        return None
    path = Path(p)
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    return path


def _model_src(key: str) -> str:
    """Return local path or HF hub id for a model key in CFG['models']."""
    if key not in CFG["models"]:
        raise KeyError(f"Unknown model key {key!r}. Choose from: {sorted(CFG['models'])}")
    return CFG["models"][key]


def load_generative(key: str):
    src = _model_src(key)
    tokenizer = AutoTokenizer.from_pretrained(src, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        src,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
    return tokenizer, model


def load_encoder(key: str):
    src = _model_src(key)
    tokenizer = AutoTokenizer.from_pretrained(src)
    model = AutoModel.from_pretrained(src)
    return tokenizer, model


def free_model(model):
    del model
    torch.cuda.empty_cache()


_pool_path = _resolve_cfg_path(CFG["pool_full"])
with open(_pool_path, "rb") as _f:
    cui_pool = pickle.load(_f)

_n_cuis = cui_pool.get("n_cuis", len(cui_pool.get("cuis", {})))
print(f"Config: {CONFIG_PATH}")
print(f"Loaded CUI pool from: {_pool_path}")
print(f"Pool type: {cui_pool.get('pool_type', 'unknown')} | unique CUIs: {_n_cuis:,}")
if _n_cuis < 10_000:
    print("WARNING: CUI count looks like the MeSH subset (~1,201), not full UMLS.")
else:
    print("Confirmed: full_umls-scale CUI pool loaded.")


/home/s224858267/.conda/envs/torch_gpu/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_ROOT: /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs
CONFIG_PATH:  /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/config.json


Config: /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/config.json
Loaded CUI pool from: /home/s224858267/data/umls/pools/cui_pool_full_umls.pkl
Pool type: full_umls | unique CUIs: 3,341,331
Confirmed: full_umls-scale CUI pool loaded.


In [2]:
# Loud fail if wrong pool loaded (mesh subset ~1.2k CUIs)
_n_cuis = int(cui_pool.get("n_cuis", len(cui_pool.get("cuis", {}))))
print(f"Setup pool_type={cui_pool.get('pool_type')} | unique CUIs={_n_cuis:,}")
assert _n_cuis > 3_000_000, (
    f"Wrong CUI pool loaded: n_cuis={_n_cuis:,} (expected full_umls > 3,000,000). "
    f"Check CFG['pool_full'] and re-run Setup."
)
print("ASSERT OK: full_umls-scale pool loaded.")


Setup pool_type=full_umls | unique CUIs=3,341,331
ASSERT OK: full_umls-scale pool loaded.


# RQ3 — Within-architecture matched pairs (generative)

**Goal:** Test whether biomedical pretraining reduces semantic entropy when only
domain differs (matched architecture).

| Pair | Biomedical | General comparator |
|---|---|---|
| Pair 2 (primary) | BioMistral-7B | Mistral-7B-Instruct-v0.1 |
| Pair 3 (convergent) | Llama3-OpenBioLLM-8B | Meta-Llama-3-8B-Instruct |
| Pair 1 (reuse only) | BioBERT | BERT-base — from RQ1 Part 2 encoders; **not** re-run |

**Datasets:** MedMentions (RQ1 perturbations), BioASQ, SQuAD 2.0. ShARe excluded.

**Batch note:** One generative model at a time (`free_model` between loads). Safe under
`nbconvert --execute` on the `torch_gpu` kernel. Does **not** regenerate perturbations.


## 1) Paths, pairs, TOP_K, and variant tables

Reads accepted perturbations from existing artifacts. TOP_K from
`outputs/rq1/k_selection.json` when present (else 50, matching Part-2 default).


In [3]:
# === RQ3 matched pairs: config / variants ===================================
import sys
import time
import gc
import re
from collections import Counter, defaultdict
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

def _log(msg: str):
    print(f"[{datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}] {msg}")
    sys.stdout.flush()

OUT_DIR = PROJECT_ROOT / "outputs" / "rq3"
INTER_DIR = OUT_DIR / "intermediate"
TAB_DIR = OUT_DIR / "tables"
for d in (OUT_DIR, INTER_DIR, TAB_DIR):
    d.mkdir(parents=True, exist_ok=True)

ENTROPY_OUT = OUT_DIR / "entropy_matched_pairs.csv"
RAW_DIR = INTER_DIR / "matched_pair_raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# TOP_K: prefer k_selection from RQ1 Part 2 amendment
_ksel_path = _resolve_cfg_path(CFG.get("k_selection", "outputs/rq1/k_selection.json"))
if _ksel_path is not None and Path(_ksel_path).exists():
    with open(_ksel_path, "r", encoding="utf-8") as _f:
        _ksel = json.load(_f)
    TOP_K = int(_ksel.get("k_selected", 50))
    _log(f"TOP_K={TOP_K} from {_ksel_path} (rule_satisfied={_ksel.get('rule_satisfied')})")
else:
    TOP_K = 50
    _log(f"TOP_K={TOP_K} (fallback; k_selection.json not found)")

MIN_FORM_LEN = 3
CONFIDENCE_THRESHOLD = 0.70
SAPBERT_ID = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
EMB_DIR = Path.home() / "data" / "umls" / "embeddings" / "sapbert_full_len3"
UNASSIGNED = "UNASSIGNED"
MAX_NEW_TOKENS = 32
MAX_PERTS = 8  # original + up to 8 accepted perts

# Matched generative pairs (config.json keys)
PAIRS = [
    {
        "pair": "pair2_biomistral_vs_mistral",
        "biomedical": {"key": "biomistral", "model_name": "BioMistral-7B", "domain": "biomedical"},
        "general": {"key": "mistral", "model_name": "Mistral-7B-Instruct-v0.1", "domain": "general"},
    },
    {
        "pair": "pair3_openbiollm_vs_llama3",
        "biomedical": {"key": "openbiollm", "model_name": "Llama3-OpenBioLLM-8B", "domain": "biomedical"},
        "general": {"key": "llama3", "model_name": "Meta-Llama-3-8B-Instruct", "domain": "general"},
    },
]
GEN_MODELS = []
for p in PAIRS:
    GEN_MODELS.append(p["biomedical"])
    GEN_MODELS.append(p["general"])

# ---- Build variant tables (original + accepted perturbations) ----
def _norm_cui(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return UNASSIGNED
    s = str(x).strip()
    if s.startswith("UMLS:"):
        s = s[5:]
    if s in {"", "NA", "nan", "None", UNASSIGNED}:
        return UNASSIGNED
    return s

# Required input CSVs (assert before any load — fail loudly with exact path)
_mm_inst = PROJECT_ROOT / "outputs" / "rq1" / "intermediate" / "rq1_sampled_instances.csv"
_mm_pert = PROJECT_ROOT / "outputs" / "rq1" / "intermediate" / "rq1_validated_perturbations.csv"
_bio_inst = PROJECT_ROOT / "outputs" / "rq3" / "intermediate" / "rq3_bioasq_instances.csv"
_bio_pert = PROJECT_ROOT / "outputs" / "rq3" / "intermediate" / "rq3_bioasq_perturbations.csv"
_sq_inst = PROJECT_ROOT / "outputs" / "rq3" / "intermediate" / "rq3_squad_instances.csv"
_sq_pert = PROJECT_ROOT / "outputs" / "rq3" / "intermediate" / "rq3_squad_perturbations.csv"
_cadec_inst = PROJECT_ROOT / "outputs" / "rq3" / "intermediate" / "rq3_cadec_instances.csv"
_cadec_pert = PROJECT_ROOT / "outputs" / "rq3" / "intermediate" / "rq3_cadec_perturbations.csv"

_REQUIRED_INPUT_CSVS = [
    ("MedMentions instances", _mm_inst),
    ("MedMentions perturbations", _mm_pert),
    ("BioASQ instances", _bio_inst),
    ("BioASQ perturbations", _bio_pert),
    ("SQuAD instances", _sq_inst),
    ("SQuAD perturbations", _sq_pert),
    ("CADEC instances", _cadec_inst),
    ("CADEC perturbations", _cadec_pert),
]
_missing = []
for _label, _path in _REQUIRED_INPUT_CSVS:
    print(f"Input CSV [{_label}]: {_path}")
    if not _path.is_file():
        _missing.append(str(_path))
sys.stdout.flush()
assert not _missing, (
    "Missing required input CSV(s):\n  - " + "\n  - ".join(_missing)
)
_log("ASSERT OK: config.json + all six input CSVs exist.")

# MedMentions from RQ1
print(f"Loading: {_mm_inst}")
df_mm_i = pd.read_csv(_mm_inst)
print(f"Loading: {_mm_pert}")
df_mm_p = pd.read_csv(_mm_pert)
if "accepted_final" in df_mm_p.columns:
    df_mm_p = df_mm_p[df_mm_p["accepted_final"] == True].copy()

mm_rows = []
for _, r in df_mm_i.iterrows():
    mm_rows.append({
        "dataset": "MedMentions",
        "instance_id": r["instance_id"],
        "input_variant_id": f"{r['instance_id']}_orig",
        "input_type": "original",
        "input_text": r["mention_context"] if pd.notna(r.get("mention_context")) else r.get("original_text"),
        "gold_mention": r.get("gold_mention"),
        "gold_cui": _norm_cui(r.get("gold_cui")),
    })
for iid, g in df_mm_p.groupby("instance_id"):
    g = g.head(MAX_PERTS)
    for _, r in g.iterrows():
        mm_rows.append({
            "dataset": "MedMentions",
            "instance_id": iid,
            "input_variant_id": r.get("perturbation_id", f"{iid}_pert"),
            "input_type": "perturbation",
            "input_text": r["perturbation_text"],
            "gold_mention": r.get("gold_mention"),
            "gold_cui": _norm_cui(r.get("gold_cui")),
        })
df_mm_var = pd.DataFrame(mm_rows)

def _load_rq3_dataset(name, inst_path, pert_path):
    print(f"Loading: {inst_path}")
    di = pd.read_csv(inst_path)
    print(f"Loading: {pert_path}")
    dp = pd.read_csv(pert_path)
    if "accepted_final" in dp.columns:
        dp = dp[dp["accepted_final"] == True].copy()
    rows = []
    for _, r in di.iterrows():
        rows.append({
            "dataset": name,
            "instance_id": r["instance_id"],
            "input_variant_id": f"{r['instance_id']}_orig",
            "input_type": "original",
            "input_text": r["mention_context"] if pd.notna(r.get("mention_context")) else r.get("question"),
            "gold_mention": r.get("gold_mention"),
            "gold_cui": _norm_cui(r.get("gold_cui")),
        })
    for iid, g in dp.groupby("instance_id"):
        g = g.head(MAX_PERTS)
        for j, (_, r) in enumerate(g.iterrows()):
            rows.append({
                "dataset": name,
                "instance_id": iid,
                "input_variant_id": f"{iid}_pert{j}",
                "input_type": "perturbation",
                "input_text": r["perturbation_text"],
                "gold_mention": r.get("gold_mention"),
                "gold_cui": _norm_cui(r.get("gold_cui")),
            })
    return pd.DataFrame(rows)

df_bio_var = _load_rq3_dataset("BioASQ", _bio_inst, _bio_pert)
df_sq_var = _load_rq3_dataset("SQuAD", _sq_inst, _sq_pert)
df_cadec_var = _load_rq3_dataset("CADEC", _cadec_inst, _cadec_pert)

df_variants = pd.concat([df_mm_var, df_bio_var, df_sq_var, df_cadec_var], ignore_index=True)
_log("Variant counts by dataset (original + accepted perts, max 8 perts):")
print(df_variants.groupby(["dataset", "input_type"]).size().unstack(fill_value=0).to_string())
sys.stdout.flush()
_n_var = df_variants.groupby(["dataset", "instance_id"]).size()
_log(f"Variants/instance: mean={_n_var.mean():.2f} min={_n_var.min()} max={_n_var.max()}")
# Expect ~9 when 8 perts accepted; report if lower
_log(f"Total variant rows={len(df_variants):,} | instances={df_variants[['dataset','instance_id']].drop_duplicates().shape[0]:,}")


[2026-07-28 01:07:29 UTC] TOP_K=1000 from /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq1/k_selection.json (rule_satisfied=False)


Input CSV [MedMentions instances]: /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq1/intermediate/rq1_sampled_instances.csv
Input CSV [MedMentions perturbations]: /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq1/intermediate/rq1_validated_perturbations.csv
Input CSV [BioASQ instances]: /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq3/intermediate/rq3_bioasq_instances.csv
Input CSV [BioASQ perturbations]: /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq3/intermediate/rq3_bioasq_perturbations.csv
Input CSV [SQuAD instances]: /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq3/intermediate/rq3_squad_instances.csv
Input CSV [SQuAD perturbations]: /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq3/intermediate/rq3_squad_perturbations.csv


[2026-07-28 01:07:29 UTC] ASSERT OK: config.json + all six input CSVs exist.


Loading: /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq1/intermediate/rq1_sampled_instances.csv
Loading: /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq1/intermediate/rq1_validated_perturbations.csv
Loading: /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq3/intermediate/rq3_bioasq_instances.csv
Loading: /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq3/intermediate/rq3_bioasq_perturbations.csv


Loading: /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq3/intermediate/rq3_squad_instances.csv

Loading: /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq3/intermediate/rq3_squad_perturbations.csv
[2026-07-28 01:07:30 UTC] Variant counts by dataset (original + accepted perts, max 8 perts):


input_type   original  perturbation
dataset                            
BioASQ            150           456
MedMentions       550          2661
SQuAD             200           628


[2026-07-28 01:07:30 UTC] Variants/instance: mean=5.16 min=1 max=9


[2026-07-28 01:07:30 UTC] Total variant rows=4,645 | instances=900


## 2) Load cached SapBERT + FAISS index (RQ1 Part 2 protocol)

Reuses `~/data/umls/embeddings/sapbert_full_len3/`. Five-rule assignment on the
**generated concept text** (generative model output): exact match → FAISS top-k →
ST21pv → confidence ≥ 0.70 → frequency tiebreak. Length guard on pool forms (≥3,
has alpha) already baked into that cache.


In [4]:
# === RQ3: SapBERT + FAISS linker (reuse Part-2 cache) ========================
import faiss
from transformers import AutoModel, AutoTokenizer

_emb_npy = EMB_DIR / "embeddings.npy"
_forms_json = EMB_DIR / "surface_forms.json"
_pairs_json = EMB_DIR / "cui_form_pairs.json"
_index_path = EMB_DIR / "faiss.index"
for p in (_emb_npy, _forms_json, _pairs_json, _index_path):
    if not p.exists():
        raise FileNotFoundError(
            f"Missing Part-2 embedding cache file: {p}. "
            f"Run RQ1_PART2_full_umls_pool.ipynb P2.1–P2.3 first."
        )

_log(f"Loading form embeddings / FAISS from {EMB_DIR}")
_form_embeddings = np.load(_emb_npy)
with open(_forms_json, "r", encoding="utf-8") as _f:
    _unique_forms = json.load(_f)
with open(_pairs_json, "r", encoding="utf-8") as _f:
    _form_cui_pairs = [tuple(x) for x in json.load(_f)]
_faiss_index = faiss.read_index(str(_index_path))

assert len(_unique_forms) == _form_embeddings.shape[0] == _faiss_index.ntotal, (
    f"Alignment broken: forms={len(_unique_forms)} emb={_form_embeddings.shape[0]} "
    f"faiss={_faiss_index.ntotal}"
)
_log(f"FAISS ntotal={_faiss_index.ntotal:,} dim={_form_embeddings.shape[1]} TOP_K={TOP_K}")

_form_to_cuis = defaultdict(set)
for _c, _f in _form_cui_pairs:
    _form_to_cuis[_f].add(_c)
_form_index = {f: i for i, f in enumerate(_unique_forms)}
_exact_index = defaultdict(set)
for _f, _cuis in _form_to_cuis.items():
    _exact_index[_f.casefold()].update(_cuis)

_raw_cuis = cui_pool["cuis"]
_cui_st21pv = {c: bool(rec.get("st21pv", False)) for c, rec in _raw_cuis.items()}
_cui_n_forms = {c: len(rec.get("surface_forms", ())) for c, rec in _raw_cuis.items()}

_device = "cuda" if torch.cuda.is_available() else "cpu"
_log(f"Loading SapBERT query encoder on {_device}: {SAPBERT_ID}")
_sap_tok = AutoTokenizer.from_pretrained(SAPBERT_ID)
_sap_mdl = AutoModel.from_pretrained(SAPBERT_ID)
_sap_mdl.to(_device).eval()
if _device == "cuda":
    _sap_mdl.half()

def _mean_pool(last_hidden, attn_mask):
    mask = attn_mask.unsqueeze(-1).expand(last_hidden.size()).float()
    summed = torch.sum(last_hidden * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

def embed_sapbert(texts, batch_size=64, max_len=64):
    vecs = []
    _sap_mdl.eval()
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = [str(t) if pd.notna(t) else "" for t in texts[i:i + batch_size]]
            enc = _sap_tok(
                batch, padding=True, truncation=True, max_length=max_len, return_tensors="pt"
            )
            enc = {k: v.to(_device) for k, v in enc.items()}
            with torch.cuda.amp.autocast(enabled=(_device == "cuda")):
                out = _sap_mdl(**enc)
                pooled = _mean_pool(out.last_hidden_state, enc["attention_mask"])
                pooled = torch.nn.functional.normalize(pooled.float(), p=2, dim=1)
            vecs.append(pooled.detach().cpu().numpy().astype(np.float32))
    arr = np.vstack(vecs)
    norms = np.linalg.norm(arr, axis=1)
    assert np.allclose(norms, 1.0, atol=1e-3), "SapBERT queries not L2-normalised"
    return arr

def assign_cui_five_rules(query_text: str, mention_text: str, q_vec: np.ndarray):
    """SapBERT+FAISS five-rule protocol (generative outputs = query text)."""
    q = (query_text or "").strip()
    rule_path = []
    D, I = _faiss_index.search(q_vec.reshape(1, -1).astype(np.float32), TOP_K)
    cand = []
    for sc, ix in zip(D[0], I[0]):
        if int(ix) < 0:
            continue
        form = _unique_forms[int(ix)]
        for cui in _form_to_cuis.get(form, ()):
            cand.append((cui, form, float(sc)))
    if not cand:
        return UNASSIGNED, 0.0, "faiss_empty"

    exact_cuis = set()
    for key in (mention_text, q):
        if key and str(key).strip():
            exact_cuis |= set(_exact_index.get(str(key).strip().casefold(), ()))
    if exact_cuis:
        exact_cand = [c for c in cand if c[0] in exact_cuis]
        if exact_cand:
            cand = exact_cand
            rule_path.append("exact_match")
        else:
            cand = [(c, str(mention_text), 1.0) for c in exact_cuis] + cand
            rule_path.append("exact_match_inject")
    else:
        rule_path.append("no_exact_match")

    st_filt = [c for c in cand if _cui_st21pv.get(c[0], False)]
    if st_filt:
        cand = st_filt
        rule_path.append("st21pv")
    else:
        rule_path.append("st21pv_skip")

    # Contextual / score already FAISS IP (=cosine); optional re-score vs form emb
    rescored = []
    for cui, form, sc in cand:
        fi = _form_index.get(form)
        if fi is None:
            rescored.append((cui, form, sc))
        else:
            rescored.append((cui, form, float(np.dot(q_vec, _form_embeddings[fi]))))
    rescored.sort(key=lambda x: x[2], reverse=True)
    rule_path.append("sapbert_cosine")

    best = rescored[0][2]
    if best < CONFIDENCE_THRESHOLD:
        rule_path.append(f"below_thresh_{CONFIDENCE_THRESHOLD}")
        return UNASSIGNED, best, "+".join(rule_path)

    top = [r for r in rescored if (best - r[2]) <= 0.02]
    top.sort(key=lambda x: (_cui_n_forms.get(x[0], 0), x[2]), reverse=True)
    rule_path.append("freq_tiebreak")
    return top[0][0], float(top[0][2]), "+".join(rule_path)

_log("SapBERT+FAISS linker ready.")


[2026-07-28 01:07:30 UTC] Loading form embeddings / FAISS from /home/s224858267/data/umls/embeddings/sapbert_full_len3


[2026-07-28 01:08:15 UTC] FAISS ntotal=7,653,278 dim=768 TOP_K=1000


[2026-07-28 01:08:53 UTC] Loading SapBERT query encoder on cuda: cambridgeltl/SapBERT-from-PubMedBERT-fulltext



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Loading weights:  24%|██▍       | 48/199 [00:00<00:00, 469.64it/s]


Loading weights:  61%|██████▏   | 122/199 [00:00<00:00, 606.60it/s]


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 741.43it/s]

[2026-07-28 01:08:59 UTC] SapBERT+FAISS linker ready.


## 3) Generative inference (one model at a time) + CUI mapping

Zero-shot, bf16, greedy (`do_sample=False`, T=0). Writes resumable raw CSVs under
`outputs/rq3/intermediate/matched_pair_raw/`. Calls `free_model()` between models.


In [5]:
# === RQ3: generative inference + CUI link ===================================
def generate_concept(text: str, tokenizer, model) -> str:
    prompt = (
        f"<s>[INST] Identify the primary medical concept in the following "
        f"clinical text. Reply with only the concept name.\n\n"
        f"Text: {text} [/INST]"
    )
    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512, padding=True)
    try:
        first_dev = next(model.parameters()).device
        enc = {k: v.to(first_dev) for k, v in enc.items()}
    except StopIteration:
        enc = {k: v.to("cuda:0") for k, v in enc.items()}
    gen_kwargs = dict(
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,  # greedy, T=0
            pad_token_id=tokenizer.pad_token_id,
        )
    # Some transformers versions reject temperature when do_sample=False
    with torch.no_grad():
        out = model.generate(**enc, **gen_kwargs)
    decoded = tokenizer.decode(out[0], skip_special_tokens=True).strip()
    # Strip prompt echo for causal LMs
    if "[/INST]" in decoded:
        decoded = decoded.split("[/INST]")[-1].strip()
    elif prompt in decoded:
        decoded = decoded.replace(prompt, "").strip()
    return decoded[:200]


def run_one_generative_model(spec: dict) -> Path:
    """Inference + CUI assignment for one model. Resumable via raw CSV."""
    key, model_name = spec["key"], spec["model_name"]
    out_path = RAW_DIR / f"raw_{key}.csv"
    if out_path.exists() and out_path.stat().st_size > 0:
        _log(f"SKIP inference {model_name} — exists {out_path.name}")
        return out_path

    src = _model_src(key)
    _log(f"LOAD generative {model_name} from {src}")
    t0 = time.perf_counter()
    tokenizer, model = load_generative(key)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    rows = []
    texts = df_variants["input_text"].fillna("").astype(str).tolist()
    for i, row in tqdm(df_variants.iterrows(), total=len(df_variants), desc=model_name):
        text = str(row["input_text"]) if pd.notna(row["input_text"]) else ""
        try:
            gen = generate_concept(text, tokenizer, model)
        except Exception as e:
            _log(f"WARN generate failed {model_name} row={i}: {e}")
            gen = ""
        rows.append({
            "dataset": row["dataset"],
            "instance_id": row["instance_id"],
            "input_variant_id": row["input_variant_id"],
            "input_type": row["input_type"],
            "input_text": text,
            "gold_mention": row["gold_mention"],
            "gold_cui": row["gold_cui"],
            "generated_answer": gen,
            "model_name": model_name,
            "model_key": key,
            "domain": spec["domain"],
        })
        if (len(rows) % 200) == 0:
            _log(f"  {model_name}: {len(rows)}/{len(df_variants)} elapsed={time.perf_counter()-t0:.0f}s")

    df_raw = pd.DataFrame(rows)
    _log(f"CUI-link {model_name} generations with SapBERT+FAISS TOP_K={TOP_K}")
    q_vecs = embed_sapbert(df_raw["generated_answer"].fillna("").astype(str).tolist(), batch_size=64)
    cuis, scs, paths = [], [], []
    for i, r in df_raw.iterrows():
        cui, sc, path = assign_cui_five_rules(
            str(r["generated_answer"]),
            str(r["gold_mention"]) if pd.notna(r["gold_mention"]) else "",
            q_vecs[i],
        )
        cuis.append(_norm_cui(cui))
        scs.append(sc)
        paths.append(path)
    df_raw["predicted_cui"] = cuis
    df_raw["confidence"] = scs
    df_raw["assign_rule_path"] = paths
    df_raw["accuracy_correct"] = [
        int(p == g) if p != UNASSIGNED and g != UNASSIGNED else 0
        for p, g in zip(df_raw["predicted_cui"], df_raw["gold_cui"])
    ]
    df_raw.to_csv(out_path, index=False)
    _log(
        f"DONE {model_name}: rows={len(df_raw)} UNASSIGNED="
        f"{(df_raw['predicted_cui']==UNASSIGNED).mean():.1%} "
        f"acc={df_raw['accuracy_correct'].mean():.3f} "
        f"elapsed={time.perf_counter()-t0:.0f}s -> {out_path.name}"
    )

    free_model(model)
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    _log(f"Freed {model_name}")
    return out_path


# Run generative models sequentially
for spec in GEN_MODELS:
    run_one_generative_model(spec)

_log("All generative models processed (or skipped).")


[2026-07-28 01:08:59 UTC] SKIP inference BioMistral-7B — exists raw_biomistral.csv


[2026-07-28 01:08:59 UTC] LOAD generative Mistral-7B-Instruct-v0.1 from /home/s224858267/data/models/Mistral-7B-Instruct-v0.1


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!



Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]


Loading weights:   0%|          | 1/291 [00:00<00:47,  6.15it/s]

...

[2026-07-28 03:08:14 UTC] CUI-link Meta-Llama-3-8B-Instruct generations with SapBERT+FAISS TOP_K=1000



/tmp/ipykernel_1691207/953950884.py:66: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(_device == "cuda")):


[2026-07-28 03:21:08 UTC] DONE Meta-Llama-3-8B-Instruct: rows=4645 UNASSIGNED=50.7% acc=0.044 elapsed=4491s -> raw_llama3.csv


[2026-07-28 03:21:16 UTC] Freed Meta-Llama-3-8B-Instruct


[2026-07-28 03:21:16 UTC] All generative models processed (or skipped).


## 4) Semantic entropy per (instance, model, dataset)

**Entropy rules (documented — apply to CADEC and all other datasets, including MedMentions):**

1. **Per-instance normalisation.** For an instance with \(m\) accepted perturbations,
   the original plus those variants yield \(m+1\) model outputs. Shannon entropy
   \(H\) (over assigned CUIs; all-UNASSIGNED → NaN) is normalised as
   \(\hat{H} = H / \log_2(m+1)\), so \(\hat{H} \in [0,1]\) regardless of \(m\).
   Do **not** use a fixed \(\log_2(9)\).

2. **Minimum-\(m\) inclusion.** Include an instance in entropy analysis only if
   \(m \ge 3\) (i.e. ≥4 total outputs). Instances with \(m < 3\) are excluded and
   **counted**. The same \(m \ge 3\) rule is applied to MedMentions for consistency.

Coverage prints below report included vs excluded counts and the accepted-count
(\(\mathrm{m}\)) distribution per dataset.


In [6]:
# === RQ3: entropy assembly ==================================================
# Rules (see markdown above):
#   (1) H_norm = H / log2(m+1)  with m = # accepted perturbations for that instance
#   (2) include only if m >= 3  (same for MedMentions / BioASQ / SQuAD / CADEC)

MIN_M_ACCEPTED = 3  # require >= 4 total outputs (original + m)

def shannon_entropy(labels):
    counts = Counter(labels)
    total = sum(counts.values())
    probs = np.array([c / total for c in counts.values()], dtype=float)
    h = float(-np.sum(probs * np.log2(np.clip(probs, 1e-12, 1.0))))
    return h, counts


def entropy_from_labels(labels, n_outputs, unassigned=UNASSIGNED):
    """Shannon H over assigned CUIs; H_norm = H / log2(n_outputs) = H / log2(m+1)."""
    labels = [
        unassigned if str(lab).lower() in {"nan", "na", "none", ""} else str(lab)
        for lab in labels
    ]
    assigned = [lab for lab in labels if lab != unassigned]
    n_variants = int(n_outputs)
    m = n_variants - 1  # accepted perturbations (original + m)
    if len(assigned) == 0:
        return {
            "n_clusters": 0,
            "semantic_entropy": np.nan,
            "normalised_entropy": np.nan,
            "dominant_cluster": unassigned,
            "all_unassigned": True,
            "n_variants": n_variants,
            "n_assigned": 0,
            "m_accepted": m,
        }
    h, counts = shannon_entropy(assigned)
    # Per-instance normalisation by log2(m+1), NOT log2(9) and NOT log2(n_assigned)
    h_hat = h / np.log2(n_variants) if n_variants >= 2 else np.nan
    return {
        "n_clusters": len(counts),
        "semantic_entropy": h,
        "normalised_entropy": h_hat,
        "dominant_cluster": max(counts.items(), key=lambda x: x[1])[0],
        "all_unassigned": False,
        "n_variants": n_variants,
        "n_assigned": len(assigned),
        "m_accepted": m,
    }


# ---- Coverage: accepted-count (m) distribution from variant tables ----
_m_counts = (
    df_variants[df_variants["input_type"] == "perturbation"]
    .groupby(["dataset", "instance_id"])
    .size()
    .rename("m_accepted")
    .reset_index()
)
# Instances with original only (m=0) still appear in df_variants
_all_inst = df_variants[["dataset", "instance_id"]].drop_duplicates()
_m_counts = _all_inst.merge(_m_counts, on=["dataset", "instance_id"], how="left")
_m_counts["m_accepted"] = _m_counts["m_accepted"].fillna(0).astype(int)
_m_counts["included"] = _m_counts["m_accepted"] >= MIN_M_ACCEPTED

_log("Entropy inclusion rule: m_accepted >= 3 (total outputs = m+1 >= 4)")
_log("Per-instance H_norm = H / log2(m+1)")
print("\n===== Accepted-count (m) distribution & inclusion =====")
for ds, g in _m_counts.groupby("dataset"):
    n_inc = int(g["included"].sum())
    n_exc = int((~g["included"]).sum())
    print(f"\n{ds}: included={n_inc:,}  excluded(m<3)={n_exc:,}  total={len(g):,}")
    print("  m distribution:")
    print(g["m_accepted"].value_counts().sort_index().to_string().replace("\n", "\n  "))
sys.stdout.flush()

_included_ids = {
    ds: set(g.loc[g["included"], "instance_id"].astype(str))
    for ds, g in _m_counts.groupby("dataset")
}

# Pair lookup
_name_to_pair = {}
_name_to_domain = {}
for p in PAIRS:
    for side in ("biomedical", "general"):
        _name_to_pair[p[side]["model_name"]] = p["pair"]
        _name_to_domain[p[side]["model_name"]] = p[side]["domain"]

ent_rows = []
_excl_model = Counter()
for spec in GEN_MODELS:
    path = RAW_DIR / f"raw_{spec['key']}.csv"
    if not path.is_file():
        _log(f"SKIP entropy {spec['model_name']}: missing {path}")
        continue
    df_raw = pd.read_csv(path)
    for (ds, iid), grp in df_raw.groupby(["dataset", "instance_id"]):
        iid_s = str(iid)
        # Prefer m from input_type counts; fall back to len(grp)-1
        if "input_type" in grp.columns:
            m = int((grp["input_type"] == "perturbation").sum())
            n_outputs = m + int((grp["input_type"] == "original").sum())
            if n_outputs == 0:
                n_outputs = len(grp)
                m = max(n_outputs - 1, 0)
        else:
            n_outputs = len(grp)
            m = max(n_outputs - 1, 0)

        if m < MIN_M_ACCEPTED:
            _excl_model[(ds, spec["model_name"])] += 1
            continue
        # Also honour variant-table inclusion set when available
        if ds in _included_ids and iid_s not in _included_ids[ds]:
            _excl_model[(ds, spec["model_name"])] += 1
            continue

        ent = entropy_from_labels(grp["predicted_cui"].tolist(), n_outputs=n_outputs)
        ent_rows.append({
            "instance_id": iid,
            "dataset": ds,
            "model_name": spec["model_name"],
            "domain": spec["domain"],
            "pair": _name_to_pair[spec["model_name"]],
            "mean_accuracy": float(grp["accuracy_correct"].mean()),
            **ent,
        })

df_ent_gen = pd.DataFrame(ent_rows)
_log(f"Generative entropy rows (m>=3 only): {len(df_ent_gen):,}")
if _excl_model:
    print("Excluded (m<3) instance×model counts:")
    for (ds, mn), n in sorted(_excl_model.items()):
        print(f"  {ds} | {mn}: {n:,}")
    sys.stdout.flush()

# Pair 1 reuse from RQ1 Part 2 encoder entropy (BioBERT vs BERT-base) — MedMentions
# Re-apply per-instance log2(m+1) normalisation and m>=3 inclusion for consistency.
_enc_path = PROJECT_ROOT / "outputs" / "rq1" / "entropy_full_umls.csv"
df_ent_enc = pd.DataFrame()
if _enc_path.exists():
    _enc = pd.read_csv(_enc_path)
    _enc = _enc[_enc["model_name"].isin(["BERT-base", "BioBERT"])].copy()
    _enc_rows = []
    _enc_excl = 0
    for _, r in _enc.iterrows():
        n_var = r.get("n_variants_fulldup", r.get("n_variants", np.nan))
        if pd.isna(n_var):
            n_var = np.nan
        else:
            n_var = int(n_var)
        m = int(n_var - 1) if pd.notna(n_var) else -1
        if m < MIN_M_ACCEPTED:
            _enc_excl += 1
            continue
        domain = "biomedical" if r["model_name"] == "BioBERT" else "general"
        all_un = bool(r["all_unassigned"]) if "all_unassigned" in r and pd.notna(r["all_unassigned"]) else pd.isna(r.get("normalised_semantic_entropy_full"))
        h = r.get("semantic_entropy_full", np.nan)
        # Re-normalise: H / log2(m+1)  (override Part-2's log2(n_assigned) if present)
        if all_un or pd.isna(h):
            h_hat = np.nan
        else:
            h_hat = float(h) / np.log2(n_var) if n_var >= 2 else np.nan
        _enc_rows.append({
            "instance_id": r["instance_id"],
            "dataset": "MedMentions",
            "model_name": r["model_name"],
            "domain": domain,
            "pair": "pair1_biobert_vs_bertbase",
            "n_clusters": r.get("n_clusters_full", np.nan),
            "semantic_entropy": h,
            "normalised_entropy": h_hat,
            "mean_accuracy": r.get("mean_accuracy_full", np.nan),
            "dominant_cluster": r.get("dominant_cluster_full", UNASSIGNED),
            "all_unassigned": all_un,
            "n_variants": n_var,
            "n_assigned": r.get("n_assigned", np.nan),
            "m_accepted": m,
        })
    df_ent_enc = pd.DataFrame(_enc_rows)
    _log(
        f"Reused Pair 1 encoder rows from {_enc_path}: {len(df_ent_enc):,} "
        f"(excluded m<3: {_enc_excl:,}; H_norm rebased to log2(m+1))"
    )
else:
    _log(f"WARNING: {_enc_path} missing — Pair 1 omitted from CSV")

df_entropy_pairs = pd.concat([df_ent_gen, df_ent_enc], ignore_index=True)

# Final coverage by dataset (entropy rows are per model — report unique instances)
print("\n===== Entropy analysis coverage (unique instances in output) =====")
if len(df_entropy_pairs):
    for ds, g in df_entropy_pairs.groupby("dataset"):
        n_inst = g["instance_id"].nunique()
        n_var_table = int((_m_counts["dataset"] == ds).sum()) if len(_m_counts) else np.nan
        n_exc_table = int(((_m_counts["dataset"] == ds) & (~_m_counts["included"])).sum()) if len(_m_counts) else np.nan
        print(f"{ds}: instances_in_entropy={n_inst:,}  "
              f"(variant-table excluded m<3={n_exc_table:,} / {n_var_table:,})")
    print("\nm_accepted among included entropy rows:")
    print(df_entropy_pairs.groupby(["dataset", "m_accepted"]).size().unstack(fill_value=0).to_string())
sys.stdout.flush()

_cols = [
    "instance_id", "dataset", "model_name", "domain", "pair",
    "n_clusters", "semantic_entropy", "normalised_entropy",
    "mean_accuracy", "dominant_cluster",
]
df_entropy_pairs.to_csv(ENTROPY_OUT, index=False)
_log(f"Wrote {ENTROPY_OUT.resolve()} rows={len(df_entropy_pairs):,}")
if len(df_entropy_pairs):
    print(df_entropy_pairs.groupby(["pair", "model_name", "dataset"]).size().unstack(fill_value=0).to_string())
sys.stdout.flush()


[2026-07-28 03:21:17 UTC] Reused Pair 1 encoder rows from /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq1/entropy_full_umls.csv: 1,100


[2026-07-28 03:21:17 UTC] Wrote /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq3/entropy_matched_pairs.csv rows=4,700


dataset                                               BioASQ  MedMentions  SQuAD
pair                        model_name                                          
pair1_biobert_vs_bertbase   BERT-base                      0          550      0
                            BioBERT                        0          550      0
pair2_biomistral_vs_mistral BioMistral-7B                150          550    200
                            Mistral-7B-Instruct-v0.1     150          550    200
pair3_openbiollm_vs_llama3  Llama3-OpenBioLLM-8B         150          550    200
                            Meta-Llama-3-8B-Instruct     150          550    200


## 5) Matched-pair statistics

Per pair: mean \(H\) domain-adapted vs comparator; one-tailed Mann-Whitney U
(\(H_{\mathrm{bio}} < H_{\mathrm{gen}}\)); rank-biserial \(r\); BH-FDR across tests;
bootstrap 95% CI (B=1000, seed 42) on the mean difference. Reported per dataset and pooled.


In [7]:
# === RQ3: matched-pair stats ================================================
from scipy import stats

B = 1000
RNG = np.random.default_rng(42)

def rank_biserial_from_u(u, n1, n2):
    # r = 1 - 2U/(n1*n2) for common MWU effect-size convention
    return 1.0 - (2.0 * u) / (n1 * n2)

def bh_fdr(pvals):
    pvals = np.asarray(pvals, dtype=float)
    n = len(pvals)
    order = np.argsort(pvals)
    ranked = np.empty(n, dtype=float)
    prev = 1.0
    for i, idx in enumerate(order[::-1], start=1):
        rank = n - i + 1
        val = min(prev, pvals[idx] * n / rank)
        ranked[idx] = val
        prev = val
    return ranked

def bootstrap_mean_diff(x_bio, x_gen, b=B, rng=RNG):
    x_bio = np.asarray(x_bio, dtype=float)
    x_gen = np.asarray(x_gen, dtype=float)
    diffs = []
    for _ in range(b):
        xb = rng.choice(x_bio, size=len(x_bio), replace=True)
        xg = rng.choice(x_gen, size=len(x_gen), replace=True)
        diffs.append(float(np.mean(xb) - np.mean(xg)))
    diffs = np.asarray(diffs)
    return float(np.mean(diffs)), float(np.quantile(diffs, 0.025)), float(np.quantile(diffs, 0.975))

stat_rows = []
_pair_defs = PAIRS + [{
    "pair": "pair1_biobert_vs_bertbase",
    "biomedical": {"model_name": "BioBERT", "domain": "biomedical"},
    "general": {"model_name": "BERT-base", "domain": "general"},
}]

datasets_plus = list(df_entropy_pairs["dataset"].dropna().unique()) + ["POOLED"]

for pdef in _pair_defs:
    pair = pdef["pair"]
    m_bio = pdef["biomedical"]["model_name"]
    m_gen = pdef["general"]["model_name"]
    for ds in datasets_plus:
        sub = df_entropy_pairs[df_entropy_pairs["pair"] == pair].copy()
        if ds != "POOLED":
            sub = sub[sub["dataset"] == ds]
        # scored only
        bio = sub[(sub["model_name"] == m_bio) & (~sub["all_unassigned"].astype(bool))]["normalised_entropy"].astype(float).dropna()
        gen = sub[(sub["model_name"] == m_gen) & (~sub["all_unassigned"].astype(bool))]["normalised_entropy"].astype(float).dropna()
        if len(bio) < 5 or len(gen) < 5:
            _log(f"SKIP stats {pair} / {ds}: n_bio={len(bio)} n_gen={len(gen)}")
            continue
        # one-tailed: bio < gen
        u, p_two = stats.mannwhitneyu(bio, gen, alternative="less")
        r_rb = rank_biserial_from_u(u, len(bio), len(gen))
        mean_bio, mean_gen = float(bio.mean()), float(gen.mean())
        d_mean = mean_bio - mean_gen
        boot_mean, lo, hi = bootstrap_mean_diff(bio.values, gen.values)
        # identical-mean assertion later (pair-level pooled)
        stat_rows.append({
            "pair": pair,
            "dataset": ds,
            "model_biomedical": m_bio,
            "model_general": m_gen,
            "n_bio": int(len(bio)),
            "n_gen": int(len(gen)),
            "mean_H_bio": mean_bio,
            "mean_H_gen": mean_gen,
            "mean_diff_bio_minus_gen": d_mean,
            "mannwhitney_U": float(u),
            "mannwhitney_p_onetail": float(p_two),
            "rank_biserial_r": float(r_rb),
            "boot_mean_diff": boot_mean,
            "boot_ci95_low": lo,
            "boot_ci95_high": hi,
        })
        _log(
            f"{pair} | {ds}: H_bio={mean_bio:.4f} H_gen={mean_gen:.4f} "
            f"Δ={d_mean:+.4f} p={p_two:.4g} r={r_rb:.3f} "
            f"CI=[{lo:+.4f},{hi:+.4f}]"
        )

df_stats = pd.DataFrame(stat_rows)
if len(df_stats):
    df_stats["p_bh_fdr"] = bh_fdr(df_stats["mannwhitney_p_onetail"].values)
    df_stats.to_csv(TAB_DIR / "rq3_matched_pair_statistics.csv", index=False)
    _log(f"Wrote {TAB_DIR / 'rq3_matched_pair_statistics.csv'}")
    print(df_stats.to_string(index=False))
    sys.stdout.flush()
else:
    raise RuntimeError("No statistics rows produced — check inference outputs.")


[2026-07-28 03:21:17 UTC] pair2_biomistral_vs_mistral | BioASQ: H_bio=0.1080 H_gen=0.1267 Δ=-0.0186 p=0.1821 r=0.047 CI=[-0.0689,+0.0294]


[2026-07-28 03:21:17 UTC] pair2_biomistral_vs_mistral | MedMentions: H_bio=0.2109 H_gen=0.2056 Δ=+0.0053 p=0.5672 r=-0.006 CI=[-0.0239,+0.0338]


[2026-07-28 03:21:17 UTC] pair2_biomistral_vs_mistral | SQuAD: H_bio=0.1717 H_gen=0.1711 Δ=+0.0006 p=0.5173 r=-0.002 CI=[-0.0466,+0.0510]


[2026-07-28 03:21:17 UTC] pair2_biomistral_vs_mistral | POOLED: H_bio=0.1851 H_gen=0.1848 Δ=+0.0004 p=0.4418 r=0.004 CI=[-0.0230,+0.0211]


[2026-07-28 03:21:17 UTC] pair3_openbiollm_vs_llama3 | BioASQ: H_bio=0.0028 H_gen=0.1955 Δ=-0.1927 p=7.651e-17 r=0.393 CI=[-0.2362,-0.1506]


...

[2026-07-28 03:21:17 UTC] pair1_biobert_vs_bertbase | MedMentions: H_bio=0.1272 H_gen=0.1239 Δ=+0.0034 p=0.6355 r=-0.009 CI=[-0.0261,+0.0306]


[2026-07-28 03:21:17 UTC] SKIP stats pair1_biobert_vs_bertbase / SQuAD: n_bio=0 n_gen=0


[2026-07-28 03:21:17 UTC] pair1_biobert_vs_bertbase | POOLED: H_bio=0.1272 H_gen=0.1239 Δ=+0.0034 p=0.6355 r=-0.009 CI=[-0.0276,+0.0311]


[2026-07-28 03:21:17 UTC] Wrote /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq3/tables/rq3_matched_pair_statistics.csv


                       pair     dataset     model_biomedical            model_general  n_bio  n_gen  mean_H_bio  mean_H_gen  mean_diff_bio_minus_gen  mannwhitney_U  mannwhitney_p_onetail  rank_biserial_r  boot_mean_diff  boot_ci95_low  boot_ci95_high     p_bh_fdr
pair2_biomistral_vs_mistral      BioASQ        BioMistral-7B Mistral-7B-Instruct-v0.1    149    148    0.108032    0.126661                -0.018629        10511.0           1.820771e-01         0.046708       -0.017631      -0.068902        0.029427 3.641543e-01
pair2_biomistral_vs_mistral MedMentions        BioMistral-7B Mistral-7B-Instruct-v0.1    549    539    0.210884    0.205555                 0.005329       148771.5           5.672080e-01        -0.005515        0.005264      -0.023890        0.033834 6.355340e-01
pair2_biomistral_vs_mistral       SQuAD        BioMistral-7B Mistral-7B-Instruct-v0.1    198    190    0.171719    0.171093                 0.000626        18851.5           5.173370e-01        -0.002206     

## 6) Assertions — pair means must differ


In [8]:
# === RQ3: assertions ========================================================
_log("ASSERT: within each pair, biomedical vs general mean H must not be identical")
for pair, g in df_entropy_pairs.groupby("pair"):
    scored = g[~g["all_unassigned"].astype(bool)]
    means = scored.groupby("model_name")["normalised_entropy"].mean()
    _log(f"  {pair}: {means.round(6).to_dict()}")
    vals = means.dropna().values
    if len(vals) >= 2 and abs(float(vals.max() - vals.min())) < 1e-12:
        raise AssertionError(
            f"BUG: identical mean H within {pair}: {means.to_dict()}. "
            f"Known bug signature if CUI mapping collapsed across models."
        )
_log("ASSERT OK: all pairs have distinct mean H across the two models.")

# Pool sanity again
assert int(cui_pool.get("n_cuis", 0)) > 3_000_000
_log("All RQ3 matched-pair assertions passed.")
_log(f"Primary outputs:\n  {ENTROPY_OUT}\n  {TAB_DIR / 'rq3_matched_pair_statistics.csv'}")


[2026-07-28 03:21:17 UTC] ASSERT: within each pair, biomedical vs general mean H must not be identical


[2026-07-28 03:21:17 UTC]   pair1_biobert_vs_bertbase: {'BERT-base': 0.123853, 'BioBERT': 0.127233}


[2026-07-28 03:21:17 UTC]   pair2_biomistral_vs_mistral: {'BioMistral-7B': 0.185126, 'Mistral-7B-Instruct-v0.1': 0.184775}


[2026-07-28 03:21:17 UTC]   pair3_openbiollm_vs_llama3: {'Llama3-OpenBioLLM-8B': 0.032714, 'Meta-Llama-3-8B-Instruct': 0.221802}


[2026-07-28 03:21:17 UTC] ASSERT OK: all pairs have distinct mean H across the two models.


[2026-07-28 03:21:17 UTC] All RQ3 matched-pair assertions passed.


[2026-07-28 03:21:17 UTC] Primary outputs:
  /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq3/entropy_matched_pairs.csv
  /home/s224858267/projects/Measuring-Semantic-Stability-in-Clinical-LLMs/outputs/rq3/tables/rq3_matched_pair_statistics.csv
